## Run the cell below to load the required libraries

In [ ]:
# Importing libraries
from qsarmodelingpy.ops import OPS
from qsarmodelingpy.cross_validation_class import CrossValidation
from qsarmodelingpy.filter import variance_cut, correlation_cut
from qsarmodelingpy import lj_cut as lj
from qsarmodelingpy.validate_yr_lno import validate
import os
import ipywidgets as widgets
from IPython.display import display
from draw_widgets import DrawWidgets
from ipyfilechooser import FileChooser

dw = DrawWidgets()

## Run the cell below in order to generate the buttons to load the matrices 

In [ ]:
fileX = widgets.FileUpload(
    accept="",  # Accepted file extension e.g. '.txt', '.pdf', 'image/*', 'image/*,.pdf'
    multiple=False,  # True to accept multiple files upload else False
    description="X matrix",
)
display(fileX)
filey = widgets.FileUpload(
    accept="",  # Accepted file extension e.g. '.txt', '.pdf', 'image/*', 'image/*,.pdf'
    multiple=False,  # True to accept multiple files upload else False
    description="y vector",
)
display(filey)

## Run the cell below in order to adequate the matrices to the correct format used for calculations

In [ ]:
df = dw.mountMatrix(fileX)
y = dw.mountyvector(filey)

## Run the cell below to choose the OPS parameters. The values pre-selected are the optimal or default values.

In [ ]:
var_cut_widget = dw.drawFloatSlider(
    value=0.1, min=0, max=1, description="", width="150pt"
)
# display(var_cut_widget)
display(widgets.HBox([widgets.Label("Variance cut:"), var_cut_widget]))

corr_cut_widget = dw.drawFloatSlider(
    value=0.3, min=0, max=1, description="", width="150pt"
)
# display(corr_cut_widget)
display(widgets.HBox([widgets.Label("Correlation cut:"), corr_cut_widget]))

nLVOPS_widget = dw.drawIntSlider(
    value=min(df.shape) / 2,
    min=1,
    max=min(df.shape),
    description="",
    width="150pt",  # OPS
)
# display(nLVOPS_widget)
display(
    widgets.HBox(
        [widgets.Label("Number of latent variables for OPS algorithm:"), nLVOPS_widget]
    )
)

nLVModel_widget = dw.drawIntSlider(
    value=int(len(df) / 5), min=1, max=min(df.shape), description="", width="150pt"
)
# display(nLVModel_widget)
display(
    widgets.HBox(
        [widgets.Label("Number of latent variables for the model:"), nLVModel_widget]
    )
)

opsWindow_widget = dw.drawIntSlider(
    value=1, min=1, max=df.shape[1] / 2, description="", width="150pt"
)
# display(opsWindow_widget)
display(widgets.HBox([widgets.Label("Initial OPS window:"), opsWindow_widget]))

opsIncrement_widget = dw.drawIntSlider(
    value=1,
    min=1,
    max=df.shape[1] - opsWindow_widget.value,
    description="",
    width="150pt",
)
# display(opsIncrement_widget)
display(
    widgets.HBox([widgets.Label("Increment for each OPS step:"), opsIncrement_widget])
)

percentage_widget = dw.drawIntSlider(
    value=50, min=1, max=100, description="", width="150pt"
)
# display(percentage_widget)
display(
    widgets.HBox(
        [
            widgets.Label("Maximum percentage of variables to be selected:"),
            percentage_widget,
        ]
    )
)

nModels_widget = dw.drawIntSlider(
    value=10, min=1, max=100, description="", width="150pt"
)
# display(nModels_widget)
display(
    widgets.HBox([widgets.Label("Maximum number of models to save:"), nModels_widget])
)

yr_crit_widget = dw.drawFloatSlider(
    value=0.3, min=0, max=1, description="", width="150pt"
)
# display(yr_crit_widget)
display(
    widgets.HBox(
        [
            widgets.Label("Criterion to aprove a model in y-randomization:"),
            yr_crit_widget,
        ]
    )
)

lno_crit_widget = dw.drawFloatSlider(
    value=0.1, min=0, max=1, description="", width="150pt"
)
# display(lno_crit_widget)
display(
    widgets.HBox(
        [
            widgets.Label("Criterion to aprove a model in leave-N-out test:"),
            lno_crit_widget,
        ]
    )
)

autoscale_widget = widgets.RadioButtons(
    options=["Yes", "No"],
    #     value='pineapple',
    description="Autoscale?",
    disabled=False,
)
display(autoscale_widget)

MIF_transform_widget = widgets.RadioButtons(
    options=["Yes", "No"],
    #     value='pineapple',
    description="",
    disabled=False,
    width="350pt",
)
# display(MIF_transform_widget)
widgets.HBox([widgets.Label("Transform MIF values?"), MIF_transform_widget])

## Run the cell below to choose the output files names

In [ ]:
print(
    "Choose the directory and type the desired filename for the matrix with the selected variables for best model"
)
out_matrix_widget = FileChooser(os.getcwd())
display(out_matrix_widget)

print(
    "Choose the directory and type the desired filename to save the cross-validation results for the best models"
)
out_cv_widget = FileChooser(os.getcwd())
display(out_cv_widget)

print(
    "Choose the directory and type the desired filename to save the best models according to OPS selection"
)
out_models_widget = FileChooser(os.getcwd())
display(out_models_widget)

In [ ]:
# Open configuration file in order to look for the matrices and the parameters to run
# OPS and cross-validation
var_cut = var_cut_widget.value
corr_cut = corr_cut_widget.value
nLVOPS = nLVOPS_widget.value
nLVModel = nLVModel_widget.value
opsWindow = opsWindow_widget.value
opsIncrement = opsIncrement_widget.value
percentage = percentage_widget.value
nModels = nModels_widget.value
yr_crit = yr_crit_widget.value
lno_crit = lno_crit_widget.value
out_matrix = out_matrix_widget.selected
out_cv = out_cv_widget.selected
out_models = out_models_widget.selected
# Filtering the matrix according to the options in configuration file
dfX = lj.transform(df) if MIF_transform_widget.value == "Yes" else df
print("Dimensions of the original matrix")
print(dfX.shape)
autoscale = autoscale_widget.value == "Yes"
indVar = variance_cut(dfX.values, var_cut)
dfVar = dfX.loc[:, dfX.columns[indVar]]
print("Dimensions of the matrix after variance cut")
print(dfVar.shape)
indCorr = correlation_cut(dfVar.values, y, corr_cut)
dfCorr = dfVar.loc[:, dfVar.columns[indCorr]]
print("Dimensions of the matrix after correlation cut")
print(dfCorr.shape)
X = dfCorr.values
ops = OPS(X, y, nLVOPS, nLVModel, opsWindow, opsIncrement, percentage, nModels, True)

In [ ]:
# Single run of OPS
ops.runOPS()
ops.saveModels(out_models)
var_sel = validate(
    X, y, ops.models["var_sel"], ops.models["Q2"], yr_cut=yr_crit, lno_cut=lno_crit
)
if var_sel != []:
    dfSel = dfCorr.loc[:, dfCorr.columns[var_sel]]
    dfSel.to_csv(out_matrix, sep=";")
    cv = CrossValidation(dfSel.values, y)
    cv.saveParameters(out_cv)
else:
    print("y-randomization or LNO failed!")

In [ ]:
# Multiple runs of OPS using feed OPS
ops.feedOPS()
ops.saveModels(out_directory + "/" + out_models)
var_sel = validate(
    X, y, ops.models["var_sel"], ops.models["Q2"], yr_cut=yr_crit, lno_cut=lno_crit
)
if var_sel != []:
    dfSel = dfCorr.loc[:, dfCorr.columns[var_sel]]
    dfSel.to_csv(out_directory + "/" + out_matrix, sep=";")
    cv = CrossValidation(dfSel.values, y)
    cv.saveParameters(out_directory + "/" + out_cv)
else:
    print("y-randomization or LNO failed!")